In [1]:
from datetime import datetime
import pybamm
import pandas as pd
import plotly.graph_objects as go
from netlist_utils import setup_circuit
from battery import LiionBatteryPack, PybammBattery, CLCBattery
import vessim as vs

import nest_asyncio
nest_asyncio.apply()

In [2]:
class MockActor(vs.Actor):
    def __init__(self, name: str = "mock", step_size = None, p = 0):
        super().__init__(name, step_size)
        self.power = p

    def p(self, now: datetime) -> float:
        return self.power

In [3]:
NS = 4
NP = 4
STEP_SIZE = 60
INITIAL_SOC = 1

In [4]:
simple_battery = vs.SimpleBattery(capacity=18.2 * NP * NS, initial_soc=INITIAL_SOC)
clc_battery = CLCBattery(number_of_cells=NS*NP, initial_soc=INITIAL_SOC)
pybamm_battery = PybammBattery(
    model=pybamm.lithium_ion.SPMe(options={"operating mode": "power", "calculate discharge energy": "true"}),
    initial_soc=INITIAL_SOC,
    number_of_cells=NS*NP,
    parameter_values=pybamm.ParameterValues("Chen2020"),
    output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
)
liion_battery_pack = LiionBatteryPack(
    model = pybamm.lithium_ion.SPMe(),
    netlist = setup_circuit(Np=NP, Ns=NS, Rb=1.5e-3, Rc=1e-2, Ri=5e-2, V=4.0, I=5.0),
    step_size=STEP_SIZE,
    initial_soc=INITIAL_SOC,
    parameter_values=pybamm.ParameterValues("Chen2020"),
)
#batteries = [liion_battery_pack]
batteries = [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [5]:
for battery in batteries:
    environment = vs.Environment(sim_start="2022-06-11 12:00:00")
    monitor = vs.Monitor()
    environment.add_microgrid(
        actors=[MockActor(p=-18)],
        controllers=[monitor],
        storage=battery,
        step_size=STEP_SIZE,
    )
    environment.run(until=15*3600)
    monitor.to_csv(f"results/{type(battery).__name__}.csv")

2024-06-13 13:56:15.931 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-0" ...
2024-06-13 13:56:15.933 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "Grid-0" ...
2024-06-13 13:56:15.934 | INFO     | mosaik.scenario:start:280 - Starting "Controller" as "Controller-0" ...
2024-06-13 13:56:15.935 | INFO     | mosaik.scenario:start:280 - Starting "Storage" as "Storage-0" ...
2024-06-13 13:56:15.938 | INFO     | mosaik.scenario:run:598 - Starting simulation.
100%|██████████| 54000/54000 [00:00<00:00, 217611.89steps/s]
2024-06-13 13:56:16.189 | INFO     | mosaik.scenario:run:646 - Simulation finished successfully.
2024-06-13 13:56:16.208 | INFO     | mosaik.scenario:start:280 - Starting "Actor" as "Actor-0" ...
2024-06-13 13:56:16.209 | INFO     | mosaik.scenario:start:280 - Starting "Grid" as "Grid-0" ...
2024-06-13 13:56:16.209 | INFO     | mosaik.scenario:start:280 - Starting "Controller" as "Controller-0" ...
2024-06-13 13:56:16.210 | INFO     | mosaik.s

In [6]:
fig = go.Figure()
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.9, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for battery in ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]:
    df = pd.read_csv(f"results/{battery}.csv", index_col=0)
    fig.add_trace(
        go.Scatter(x=[i * STEP_SIZE for i in range(len(df.index))], y=df["storage.soc"], name=battery, mode="lines")
    )
fig.show()